# 05b — Single-Task Intent Control (`single_task_intent_plain` & `single_task_intent_mlm_attn`)

**Hypothesis Tested:** When predicting the 6 communicative intent categories (`complaint, feedback, greeting, inquiry, other, suggestion`) **without joint sentiment supervision**:
1. Does removing the clean sentiment task help or hurt gold intent performance relative to Joint `MLM+attn` (`0.398` Macro-F1 in Run A; `0.410` in Run B)?
2. Does domain MLM + single-query 4-head attention pooling (`single_task_intent_mlm_attn`) beat unadapted mean pooling (`single_task_intent_plain`) on intent alone, or is intent performance capped regardless near the 45.0% silver--gold agreement ceiling?

- Exports **gold** test metrics (`results/05b_single_task_intent_*_{run}_{seed}.json`), **silver-proxy** intent metrics (in `meta["silver_proxy_intent"]`), confusion matrices (`cm_intent_*.png/.csv`), and `stage1_cache_*.npz` (`val_probs`, `test_probs`) for `05c` cascades (`Intent -> Sent`) and two-model pipelines.
- Includes a local `evaluate_and_save_compat` fallback so it runs cleanly even if cloned against an unpatched remote `eval_utils.py`.

## 1. Environment, Fallback Helper & Single Config Cell

In [ ]:
# !pip install -q transformers accelerate
# -- Colab / Repo-Root Setup --------------------------------------------------
import os, sys, subprocess
if "google.colab" in sys.modules and not os.path.exists("eval_utils.py"):
    if not os.path.exists("/content/Sinhala-joint-sentiment-and-intent"):
        subprocess.run(
            ["git", "clone", "https://github.com/ThiwankaChanditha/Sinhala-joint-sentiment-and-intent.git", "/content/Sinhala-joint-sentiment-and-intent"],
            check=True,
        )
    os.chdir("/content/Sinhala-joint-sentiment-and-intent")
elif not os.path.exists("eval_utils.py"):
    for _cand in ["..", "../..", "/content/Sinhala-joint-sentiment-and-intent"]:
        if os.path.exists(os.path.join(_cand, "eval_utils.py")):
            os.chdir(_cand)
            break
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
import gc
import math
import time
import json
import glob
import zipfile
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix, classification_report
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForMaskedLM,
    DataCollatorForLanguageModeling,
    get_linear_schedule_with_warmup,
)

from eval_utils import (
    SENT_CLASSES,
    INTENT_CLASSES,
    NS,
    NI,
    SEEDS,
    ID_TO_SENT,
    ID_TO_INTENT,
    log_environment,
    set_all_seeds,
    load_splits,
    macro_f1_present,
    compute_metrics,
    evaluate_and_save,
    save_intent_confusion_matrix,
    summarize_Across_seeds,
)

warnings.filterwarnings("ignore")

def evaluate_and_save_compat(
    notebook_name: str,
    model_name: str,
    run: str,
    seed: int,
    test_df: pd.DataFrame,
    pred_intent_id: np.ndarray,
    intent_probs: np.ndarray,
    extra_meta: dict,
    results_dir: str,
    preds_dir: str,
) -> dict:
    """Use eval_utils.evaluate_and_save if it supports pred_sent_id=None; otherwise self-contained fallback."""
    try:
        return evaluate_and_save(
            notebook_name=notebook_name,
            model_name=model_name,
            run=run,
            seed=seed,
            test_df=test_df,
            pred_sent_id=None,
            pred_intent_id=pred_intent_id,
            sent_probs=None,
            intent_probs=intent_probs,
            extra_meta=extra_meta,
            results_dir=results_dir,
            preds_dir=preds_dir,
        )
    except Exception:
        os.makedirs(results_dir, exist_ok=True)
        os.makedirs(preds_dir, exist_ok=True)
        y_sent_gold = test_df["gold_sent_id"].values
        y_int_gold  = test_df["gold_intent_id"].values
        yi_p        = np.asarray(pred_intent_id, dtype=np.int64)
        intent_acc  = float((y_int_gold == yi_p).mean())
        p, r, f1, sup = precision_recall_fscore_support(y_int_gold, yi_p, labels=list(range(NI)), zero_division=0)
        cm_int = confusion_matrix(y_int_gold, yi_p, labels=list(range(NI)))
        payload = {
            "notebook": notebook_name,
            "model": model_name,
            "run": run,
            "seed": seed,
            "n_test": len(test_df),
            "sent_acc": None,
            "sent_f1_4cls": None,
            "sent_f1_3cls": None,
            "intent_acc": intent_acc,
            "intent_f1": float(np.mean(f1)),
            "joint_acc": None,
            "sentiment_per_class": None,
            "intent_per_class": {
                INTENT_CLASSES[i]: {"precision": float(p[i]), "recall": float(r[i]), "f1": float(f1[i]), "support": int(sup[i])}
                for i in range(NI)
            },
            "intent_confusion_matrix": cm_int.tolist(),
            "meta": extra_meta,
        }
        with open(os.path.join(results_dir, f"{notebook_name}_{run}_{seed}.json"), "w", encoding="utf-8") as f:
            json.dump(payload, f, indent=2)
        pred_df = pd.DataFrame({
            "uid": test_df["uid"].values,
            "raw_text": test_df["raw_text"].values,
            "text": test_df["text"].values,
            "gold_sent_id": y_sent_gold,
            "gold_intent_id": y_int_gold,
            "gold_sentiment": [ID_TO_SENT[int(i)] for i in y_sent_gold],
            "gold_intent": [ID_TO_INTENT[int(i)] for i in y_int_gold],
            "pred_sent_id": -1,
            "pred_sentiment": "",
            "pred_intent_id": yi_p,
            "pred_intent": [ID_TO_INTENT[int(i)] for i in yi_p],
        })
        if intent_probs is not None:
            for idx, cname in enumerate(INTENT_CLASSES):
                pred_df[f"prob_intent_{cname}"] = np.round(intent_probs[:, idx], 6)
        pred_df.to_csv(os.path.join(preds_dir, f"{model_name}_{run}_{seed}.csv"), index=False)
        save_intent_confusion_matrix(
            cm_int,
            csv_path=os.path.join(results_dir, f"cm_intent_{notebook_name}_{run}_{seed}.csv"),
            png_path=os.path.join(results_dir, f"cm_intent_{notebook_name}_{run}_{seed}.png"),
            title=f"Intent CM: {model_name} ({run}, seed={seed})",
        )
        return payload

# -- Single Config Cell (Identical Hyperparameters to Ablation) ---------------
MODEL_NAME     = "xlm-roberta-base"
MAX_LEN        = 128
MLM_PROB       = 0.15
MLM_EPOCHS     = 3
MLM_BATCH      = 8
MLM_GRAD_ACC   = 2
MLM_LR         = 2e-5

FT_EPOCHS      = 10
FT_BATCH       = 8
FT_GRAD_ACC    = 2
FT_LR          = 2e-5
ENC_LR_MULT    = 0.1
WEIGHT_DECAY   = 0.01
DROPOUT        = 0.3
NUM_ATTN_HEADS = 4
PATIENCE       = 3

RUNS_TO_EXEC   = ["runA", "runB"]
SEEDS_TO_EXEC  = SEEDS
NOTEBOOK_NAME  = "05b_single_task_intent"

WORK_DIR       = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
SILVER_CSV     = None
GOLD_CSV       = None
RESULTS_DIR    = os.path.join(WORK_DIR, "results")
PREDS_DIR      = os.path.join(WORK_DIR, "preds")
ENC_DIR        = os.path.join(WORK_DIR, "mlm_encoders")
for _d in (RESULTS_DIR, PREDS_DIR, ENC_DIR):
    os.makedirs(_d, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
env_info = log_environment()
print("Using device:", DEVICE)

## 2. Dataset & Single-Task Intent Models (plain and MLM+attn)

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
mlm_collator = DataCollatorForLanguageModeling(tokenizer=tok, mlm=True, mlm_probability=MLM_PROB)

class IntentDS(Dataset):
    def __init__(self, df: pd.DataFrame):
        enc = tok(df["text"].tolist(), truncation=True, max_length=MAX_LEN)
        self.ids = enc["input_ids"]
        self.am  = enc["attention_mask"]
        self.y   = df["intent_id"].tolist()

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        return {"input_ids": self.ids[idx], "attention_mask": self.am[idx], "intent": self.y[idx]}

def collate_intent(batch):
    out = tok.pad(
        [{"input_ids": b["input_ids"], "attention_mask": b["attention_mask"]} for b in batch],
        return_tensors="pt",
    )
    out["intent"] = torch.tensor([b["intent"] for b in batch], dtype=torch.long)
    return out

class MeanPooler(nn.Module):
    def __init__(self, hidden: int):
        super().__init__()
        self.norm = nn.LayerNorm(hidden)

    def forward(self, H, mask):
        m = mask.unsqueeze(-1).float()
        return self.norm((H * m).sum(1) / m.sum(1).clamp(min=1.0))

class SingleTaskAttnPooler(nn.Module):
    def __init__(self, hidden: int, heads: int, dropout: float):
        super().__init__()
        self.attn = nn.MultiheadAttention(hidden, heads, dropout=dropout, batch_first=True)
        self.query = nn.Parameter(torch.randn(1, 1, hidden))
        self.norm = nn.LayerNorm(hidden)

    def forward(self, H, mask):
        B = H.size(0)
        kpm = (mask == 0)
        out, _ = self.attn(self.query.expand(B, -1, -1), H, H, key_padding_mask=kpm)
        return self.norm(out.squeeze(1))

class SingleTaskIntentModel(nn.Module):
    def __init__(self, enc_path: str, pooling: str = "attn"):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(enc_path)
        h = self.encoder.config.hidden_size
        self.pool = SingleTaskAttnPooler(h, NUM_ATTN_HEADS, DROPOUT) if pooling == "attn" else MeanPooler(h)
        self.int_fc = nn.Sequential(nn.Linear(h, h // 2), nn.GELU(), nn.Dropout(DROPOUT))
        self.int_head = nn.Linear(h // 2, NI)

    def forward(self, input_ids, attention_mask):
        H = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        p = self.pool(H, attention_mask)
        return self.int_head(self.int_fc(p))

@torch.no_grad()
def predict_intent_probs(model, loader) -> np.ndarray:
    model.eval()
    probs = []
    for b in loader:
        logits = model(b["input_ids"].to(DEVICE), b["attention_mask"].to(DEVICE))
        probs.append(torch.softmax(logits, dim=-1).cpu())
    return torch.cat(probs, dim=0).numpy()

def class_weights(ids, K: int) -> torch.Tensor:
    c = np.bincount(ids, minlength=K).astype(float)
    return torch.tensor(np.where(c > 0, c.sum() / (K * np.maximum(c, 1)), 0.0), dtype=torch.float)

## 3. Domain MLM Helper & 3-Seed Single-Task Intent Training Loop

In [ ]:
def get_mlm_encoder(mlm_examples, run: str, seed: int) -> str:
    candidates = [
        os.path.join(ENC_DIR, f"mlm_{run}_seed{seed}"),
        os.path.join(ENC_DIR, f"mlm_seed{seed}"),
    ]
    for c in candidates:
        if os.path.exists(os.path.join(c, "config.json")):
            print(f"Using cached domain MLM encoder: {c}")
            return c

    out = candidates[0]
    os.makedirs(out, exist_ok=True)
    set_all_seeds(seed)
    m = AutoModelForMaskedLM.from_pretrained(MODEL_NAME).to(DEVICE)
    m.train()
    g = torch.Generator()
    g.manual_seed(seed)
    loader = DataLoader(mlm_examples, batch_size=MLM_BATCH, shuffle=True, collate_fn=mlm_collator, generator=g)
    opt = torch.optim.AdamW(m.parameters(), lr=MLM_LR, weight_decay=WEIGHT_DECAY)
    total = MLM_EPOCHS * math.ceil(len(loader) / MLM_GRAD_ACC)
    sched = get_linear_schedule_with_warmup(opt, max(1, total // 10), total)

    for ep in range(MLM_EPOCHS):
        tl, n = 0.0, 0
        opt.zero_grad()
        for step, b in enumerate(loader):
            b = {k: v.to(DEVICE) for k, v in b.items()}
            if (b["labels"] != -100).any():
                loss = m(**b).loss
                (loss / MLM_GRAD_ACC).backward()
                tl += loss.item()
                n += 1
            if (step + 1) % MLM_GRAD_ACC == 0 or step + 1 == len(loader):
                nn.utils.clip_grad_norm_(m.parameters(), 1.0)
                opt.step()
                sched.step()
                opt.zero_grad()
        print(f"  [MLM {run} seed {seed}] epoch {ep+1}/{MLM_EPOCHS} loss {tl/max(n,1):.4f}")

    m.base_model.save_pretrained(out)
    del m, opt, sched
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return out

VARIANTS = [
    ("single_task_intent_plain",    "05b_single_task_intent_plain",    False, "mean"),
    ("single_task_intent_mlm_attn", "05b_single_task_intent_mlm_attn", True,  "attn"),
]

summaries = []

for run in RUNS_TO_EXEC:
    train_df, val_df, test_df = load_splits(run=run, silver_csv=SILVER_CSV, gold_csv=GOLD_CSV)
    mlm_texts = train_df["text"].tolist() + val_df["text"].tolist()
    assert not (set(mlm_texts) & set(test_df["text"])), "Leakage guard: test text in MLM corpus!"
    _enc = tok(mlm_texts, truncation=True, max_length=MAX_LEN, return_special_tokens_mask=True)
    mlm_examples = [
        {"input_ids": a, "attention_mask": b, "special_tokens_mask": c}
        for a, b, c in zip(_enc["input_ids"], _enc["attention_mask"], _enc["special_tokens_mask"])
    ]

    train_ds, val_ds, test_ds = IntentDS(train_df), IntentDS(val_df), IntentDS(test_df)
    val_loader  = DataLoader(val_ds,  batch_size=FT_BATCH * 2, shuffle=False, collate_fn=collate_intent)
    test_loader = DataLoader(test_ds, batch_size=FT_BATCH * 2, shuffle=False, collate_fn=collate_intent)
    W_INT = class_weights(train_df["intent_id"].values, NI).to(DEVICE)
    val_yi = val_df["intent_id"].values
    test_silver_yi = test_df["intent_id"].values if "intent_id" in test_df.columns else None

    for model_tag, nb_tag, use_mlm, pooling in VARIANTS:
        seed_payloads = []
        for seed in SEEDS_TO_EXEC:
            enc_path = get_mlm_encoder(mlm_examples, run=run, seed=seed) if use_mlm else MODEL_NAME
            set_all_seeds(seed)
            model = SingleTaskIntentModel(enc_path, pooling=pooling).to(DEVICE)
            g = torch.Generator()
            g.manual_seed(seed)
            loader = DataLoader(train_ds, batch_size=FT_BATCH, shuffle=True, collate_fn=collate_intent, generator=g)

            crit_i = nn.CrossEntropyLoss(weight=W_INT)
            head_params = [p for n, p in model.named_parameters() if not n.startswith("encoder.")]
            opt = torch.optim.AdamW(
                [{"params": model.encoder.parameters(), "lr": FT_LR * ENC_LR_MULT},
                 {"params": head_params, "lr": FT_LR}],
                weight_decay=WEIGHT_DECAY,
            )
            total = FT_EPOCHS * math.ceil(len(loader) / FT_GRAD_ACC)
            sched = get_linear_schedule_with_warmup(opt, max(1, total // 10), total)

            best_val, best_ep, best_state, bad = -1.0, 0, None, 0
            t0 = time.time()
            for ep in range(FT_EPOCHS):
                model.train()
                tl = 0.0
                opt.zero_grad()
                for step, b in enumerate(loader):
                    logits = model(b["input_ids"].to(DEVICE), b["attention_mask"].to(DEVICE))
                    loss = crit_i(logits, b["intent"].to(DEVICE))
                    (loss / FT_GRAD_ACC).backward()
                    tl += loss.item()
                    if (step + 1) % FT_GRAD_ACC == 0 or step + 1 == len(loader):
                        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                        opt.step()
                        sched.step()
                        opt.zero_grad()
                vi = predict_intent_probs(model, val_loader)
                f_i = macro_f1_present(val_yi, vi.argmax(1), NI)
                print(f"  [{model_tag} {run} s{seed}] ep {ep+1:>2} | train {tl/len(loader):.4f} | val Intent F1 {f_i:.3f}")
                if f_i > best_val:
                    best_val, best_ep, bad = f_i, ep + 1, 0
                    best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                else:
                    bad += 1
                    if bad >= PATIENCE:
                        print("  early stop")
                        break

            elapsed_s = round(time.time() - t0, 2)
            model.load_state_dict(best_state)
            vi_best = predict_intent_probs(model, val_loader)
            ti_best = predict_intent_probs(model, test_loader)
            pred_int = ti_best.argmax(1)

            cache_npz = os.path.join(PREDS_DIR, f"stage1_cache_{model_tag}_{run}_{seed}.npz")
            np.savez_compressed(
                cache_npz,
                val_probs=vi_best.astype(np.float32),
                test_probs=ti_best.astype(np.float32),
                val_pred=vi_best.argmax(1).astype(np.int64),
                test_pred=pred_int.astype(np.int64),
            )

            silver_proxy = None
            if test_silver_yi is not None:
                silver_proxy = {
                    "silver_intent_acc": round(float((test_silver_yi == pred_int).mean()), 4),
                    "silver_intent_f1": round(macro_f1_present(test_silver_yi, pred_int, NI), 4),
                }

            del model, opt, sched, best_state
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            payload = evaluate_and_save_compat(
                notebook_name=nb_tag,
                model_name=model_tag,
                run=run,
                seed=seed,
                test_df=test_df,
                pred_intent_id=pred_int,
                intent_probs=ti_best,
                extra_meta={
                    "use_mlm": use_mlm,
                    "pooling": pooling,
                    "best_epoch": best_ep,
                    "best_val_intent_f1": round(float(best_val), 4),
                    "silver_proxy_intent": silver_proxy,
                    "train_seconds": elapsed_s,
                    "stage1_cache": cache_npz,
                },
                results_dir=RESULTS_DIR,
                preds_dir=PREDS_DIR,
            )
            seed_payloads.append(payload)

            if seed == 42:
                print(f"\n--- Per-Class Gold Intent Report ({model_tag}, {run}, seed=42) ---")
                print(classification_report(test_df["gold_intent_id"].values, pred_int, target_names=INTENT_CLASSES, digits=3, zero_division=0))

        summaries.append(summarize_Across_seeds(seed_payloads))

df_sum = pd.DataFrame(summaries)
display(df_sum[["run", "model", "n_seeds", "intent_acc_str", "intent_f1_str"]])
with open(os.path.join(RESULTS_DIR, f"{NOTEBOOK_NAME}_summary.json"), "w", encoding="utf-8") as f:
    json.dump(summaries, f, indent=2)

## 4. Diagnostic Comparison vs. Joint MLM+attn & Package Outputs

In [ ]:
JOINT_INT_REF = {
    "runA": {"intent_acc": 0.454, "intent_f1": 0.398},
    "runB": {"intent_acc": 0.474, "intent_f1": 0.410},
}
for row in summaries:
    ref = JOINT_INT_REF.get(row["run"], {})
    if ref and row.get("intent_acc_mean") is not None:
        print(
            f"[{row['run']} | {row['model']}] "
            f"Gold Intent Acc={row['intent_acc_str']} (vs Joint {ref['intent_acc']:.3f}: {row['intent_acc_mean'] - ref['intent_acc']:+.3f}) | "
            f"Intent F1={row['intent_f1_str']} (vs Joint {ref['intent_f1']:.3f}: {row['intent_f1_mean'] - ref['intent_f1']:+.3f})"
        )

zip_path = os.path.join(WORK_DIR, f"{NOTEBOOK_NAME}_outputs.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for folder in (RESULTS_DIR, PREDS_DIR):
        for fp in sorted(glob.glob(os.path.join(folder, "*05b*")) + glob.glob(os.path.join(folder, "*single_task_intent*")) + glob.glob(os.path.join(folder, "stage1_cache_*"))):
            zf.write(fp, arcname=os.path.relpath(fp, WORK_DIR))
print(f"Packaged outputs into: {zip_path}")